## 查看数据

In [2]:
import pandas as pd

In [5]:
df = pd.read_csv('data/sleep_mobile_stress_dataset_15000.csv')
df.shape

(15000, 13)

In [6]:
df

,user_id,age,gender,occupation,daily_screen_time_hours,phone_usage_before_sleep_minutes,sleep_duration_hours,sleep_quality_score,stress_level,caffeine_intake_cups,physical_activity_minutes,notifications_received_per_day,mental_fatigue_score
0,1,56,Female,Designer,3.26,86,5.31,7.72,3.49,0,35,119,3.57
1,2,46,Female,Teacher,1.85,32,7.36,9.70,3.01,0,16,299,1.91
2,3,32,Female,Designer,3.04,107,4.50,6.38,5.03,0,17,21,6.05
3,4,25,Male,Software Engineer,9.00,36,6.68,5.53,10.00,0,3,220,9.92
4,5,38,Female,Teacher,3.52,56,7.57,6.69,6.71,4,92,167,5.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,14996,39,Male,Manager,8.05,45,8.91,4.33,10.00,1,40,259,10.00
14996,14997,22,Female,Student,6.44,80,6.00,5.22,10.00,2,41,231,7.54
14997,14998,51,Female,Software Engineer,7.01,78,7.27,5.37,8.66,4,93,288,7.78
14998,14999,56,Female,Software Engineer,7.89,6,5.70,5.71,9.56,1,91,47,9.98


In [9]:
df.describe()

,user_id,age,daily_screen_time_hours,phone_usage_before_sleep_minutes,sleep_duration_hours,sleep_quality_score,stress_level,caffeine_intake_cups,physical_activity_minutes,notifications_received_per_day,mental_fatigue_score
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.00000,15000.000000,15000.000000,15000.000000
mean,7500.500000,38.488467,5.501528,59.708933,6.509683,6.246362,6.980247,1.99880,59.157133,160.890467,6.873009
std,4330.271354,12.007970,2.600085,34.641858,1.452689,1.713644,2.749382,1.41459,34.525705,80.856902,2.730482
min,1.000000,18.000000,1.000000,0.000000,4.000000,1.000000,1.000000,0.00000,0.000000,20.000000,1.000000
25%,3750.750000,28.000000,3.260000,29.000000,5.260000,5.000000,4.750000,1.00000,29.000000,92.000000,4.700000
50%,7500.500000,38.000000,5.490000,60.000000,6.490000,6.250000,7.380000,2.00000,59.000000,162.000000,7.380000
75%,11250.250000,49.000000,7.760000,90.000000,7.790000,7.500000,10.000000,3.00000,89.000000,231.000000,9.450000
max,15000.000000,59.000000,10.000000,119.000000,9.000000,10.000000,10.000000,4.00000,119.000000,299.000000,10.000000


观察发现：
user_id 字段不具有特征意义，应剔除
gender 和 occupation 非数值类型，可使用 One-hot 编码
age, phone_usage_before_sleep_minutes, physical_activity_minutes, notifications_received_per_day 字段数值标准差较大，建议标准化

## 数据预处理

In [14]:
from sklearn.model_selection import train_test_split

# 剔除user_id
df_model = df.drop(columns=['user_id'])

# One-hot编码
df_model = pd.get_dummies(df_model, columns=['gender', 'occupation'], dtype=int)

# 先划分数据集
train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=42)

In [16]:
# 根据训练集训练 Scaler标准化函数
from sklearn.preprocessing import StandardScaler
import joblib

# 需要标准化的数值列
numeric_cols = ['age', 'daily_screen_time_hours', 'phone_usage_before_sleep_minutes', 
                'sleep_duration_hours', 'sleep_quality_score', 'caffeine_intake_cups', 
                'physical_activity_minutes', 'notifications_received_per_day']

scaler = StandardScaler()
scaler.fit(train_df[numeric_cols])

joblib.dump(scaler, 'models/my_scaler.pkl')

def my_scaler(df_input):
    
    data = df_input.copy()
    loaded_scaler = joblib.load('models/my_scaler.pkl')
    data[numeric_cols] = loaded_scaler.transform(data[numeric_cols])

    return data


In [17]:
train = my_scaler(train_df)
test = my_scaler(test_df)
train.to_csv('data/train_data.csv', index=False)
test.to_csv('data/test_data.csv', index=False)